# Cleanup, variable selection, and QC

In this notebook we will go through the QC procedure to select, and clean the NHANES data

## Initial set up

- Import libraries
- Read data
- Select covariates, phenotypes, and exposures

In [1]:
#### IMPORT MODULES
import nhanes as nh
import clarite
import os
import numpy as np
import pandas as pd
import warnings

#### READ DATA
paths   = nh.set_project_paths()
nhanes  = nh.NhanesData(paths[1])
weights = nh.WeightData(paths[1])
plot_paths = os.path.join(paths[2], 'Plots', 'Inspect')

#### CLEAN SURVEY WEIGHTS
weights.harmonize()

#### SUMMARY OF NHANES DATASET
clarite.describe.summarize(nhanes.data)
nhanes.print_how_many_types()

rpy2 ModuleSpec(name='rpy2', loader=<_frozen_importlib_external.SourceFileLoader object at 0x2b0a35d41c90>, origin='/storage/home/tug156/work/software/anaconda3/envs/py_clarite/lib/python3.7/site-packages/rpy2/__init__.py', submodule_search_locations=['/storage/home/tug156/work/software/anaconda3/envs/py_clarite/lib/python3.7/site-packages/rpy2'])
Running recode_values
--------------------------------------------------------------------------------
Replaced 27 values from 41,474 observations in 2 variables
-----Adjusting lipid variables-----
	ATORVASTATIN_CALCIUM = ATORVASTATIN_CALCIUM
	SIMVASTATIN = SIMVASTATIN
	PRAVASTATIN_SODIUM = PRAVASTATIN_SODIUM
	FLUVASTATIN_SODIUM = FLUVASTATIN_SODIUM
There are 1059 participants on statins

-----Dividing dataset into phenotypes, exposures and covariates-----


-----Harmonizing survey weights-----
WTSPP2YR weight, is not in data columns in Replication cohort
WTSPH2YR weight, is not in data columns in Replication cohort
WTSTH2YR weight, is not in

## Selecting participants

We'll keep participants older than 18 years, and remove those that have missing data in any of the covariates

In [2]:
#### KEEP ADULTS
nhanes.keep_adults()
#### REMOVE PARTICIPANTS WITH MISSING COVARIATES
nhanes.remove_missing_covariates()

-----Keeping only adults in the dataset-----

There are only 22624 adults out of 41474 participants

Running rowfilter_incomplete_obs
--------------------------------------------------------------------------------
Removed 3,687 of 22,624 observations (16.30%) due to NA values in any of 10 variables


## Separate discovery and replication, and males and females

In [3]:
nhanes.divide_into_cohorts()

-----Dividing into discovery/replications and female/male cohorts-----



## Variable selection and clenup

Here we will:
- Drop variables that don't have 4 year weights
- Drop any variables that are indeterminant according to the NHANES data dictionary
- Drop any non-environmental exposures (examples physical fitness)
- Adjust lipid estimates if on statin medication

### Drop variables that don't have weights

Also, some weight variables from the replication are not present, remove them as well


In [4]:
#### REMOVE VARIABLES WITHOUT WEIGHTS
nhanes.remove_missing_weights(weights)
nhanes.print_how_many_types()

-----Removing variables not in the survey weights-----
-----Matching variable names with data-----

Removing LBXAPB phenotype
Removing LBXGLT phenotype
Removing DR1BWATR exposure
Removing DR1CWATR exposure
Removing DR1TMOIS exposure
Removing DR1TSODI exposure
Removing DR1_320 exposure
Removing DR1_330 exposure
Removing DR1_330Z exposure
Removing DRDRESP exposure
Removing DRDSDT1 exposure
Removing DRDSDT2 exposure
Removing DRDSDT3 exposure
Removing DRDSDT4 exposure
Removing DRDSDT5 exposure
Removing DRDSDT6 exposure
Removing DRDSDT7 exposure
Removing DRDSDT8 exposure
Removing LBX044 exposure
Removing LBX049 exposure
Removing LBX209 exposure
Removing LBXBB1 exposure
Removing LBXBR1 exposure
Removing LBXBR2 exposure
Removing LBXBR3 exposure
Removing LBXBR4 exposure
Removing LBXBR5 exposure
Removing LBXBR6 exposure
Removing LBXBR66 exposure
Removing LBXBR7 exposure
Removing LBXBR8 exposure
Removing LBXBR9 exposure
Removing LBXEPAH exposure
Removing LBXMPAH exposure
Removing LBXPFBS exposur

## Quality control

### Categorize variables

Categorize and remove constant variables (those that have one value)

In [5]:
nhanes.categorize_variables()
nhanes.print_how_many_types()

-----Categorizing in Discovery females cohort-----
Running categorize
--------------------------------------------------------------------------------
43 of 730 variables (5.89%) are classified as constant (1 unique value).
125 of 730 variables (17.12%) are classified as binary (2 unique values).
36 of 730 variables (4.93%) are classified as categorical (3 to 6 unique values).
369 of 730 variables (50.55%) are classified as continuous (>= 15 unique values).
106 of 730 variables (14.52%) were dropped.
	106 variables had zero unique values (all NA).
51 of 730 variables (6.99%) were not categorized and need to be set manually.
	51 variables had between 6 and 15 unique values
	0 variables had >= 15 values but couldn't be converted to continuous (numeric) values

-----Removing constant variables-----
We will remove 43 constant variables
-----Categorizing in Discovery males cohort-----
Running categorize
--------------------------------------------------------------------------------
49 of 7

### Examine unknown variables

All unknown variables are continuous

In [6]:
nhanes.print_unknown_vars()
nhanes.make_unknown_continuous()

In the Discovery females cohort,there are 38 variables unknown
	DRD370UQ = # of times other unknown fish eaten
	L_GLUTAMINE_mg = L_GLUTAMINE_mg
	DRD350FQ = # of times oysters eaten in past 30 days
	SMD160 = Age started cigar smoking regularly
	LBXEND = Endrin (ng/g)
	METHIONINE_mg = METHIONINE_mg
	OMEGA_3_FATTY_ACIDS_mg = OMEGA_3_FATTY_ACIDS_mg
	SMD220 = Age started chewing tobacco regularly
	SMD190 = Age started using snuff regularly
	DRD350BQ = # of times crabs eaten in past 30 days
	URXMET = Metolachlor mercapturate (ug/L)
	LBX149 = PCB149 (ng/g)
	house_age = house age
	LBX151 = PCB151 (ng/g)
	SMD130 = Age started pipe smoking regularly
	LBX087 = PCB87 (ng/g)
	DRD370NQ = # of times sardines eaten past 30 days
	DRD370MQ = # of times salmon eaten in past 30 days
	URXUPT = Platinum, urine (ng/mL)
	quantity_drink_per_day = drink per day
	LBX189 = PCB189 (ng/g)
	BETA_CAROTENE_IU = BETA_CAROTENE_IU
	LBDBANO = Basophils number
	GLYCINE_mg = GLYCINE_mg
	DRD350HQ = # of times shrimp eaten in

### Clean up variables

- Remove variables that have less than 200 non-NA values
- Remove binary variables that have less than 200 values in a category
- Remove continuous variables with 90% of non-NA observations equal to zero

In [7]:
nhanes.clarite_filters()
nhanes.print_how_many_types()

Running in Discovery females cohort,
-----------------------------------
Running colfilter_min_n
--------------------------------------------------------------------------------
Testing 89 of 94 binary variables
	Removed 17 (19.10%) tested binary variables which had less than 200 non-null values.
Testing 27 of 28 categorical variables
	Removed 15 (55.56%) tested categorical variables which had less than 200 non-null values.
Testing 353 of 355 continuous variables
	Removed 13 (3.68%) tested continuous variables which had less than 200 non-null values.
Running colfilter_min_cat_n
--------------------------------------------------------------------------------
Testing 72 of 77 binary variables
	Removed 41 (56.94%) tested binary variables which had a category with less than 200 values.
Testing 12 of 13 categorical variables
	Removed 9 (75.00%) tested categorical variables which had a category with less than 200 values.
Running colfilter_percent_zero
----------------------------------------

### Harmonize across cohorts

Note discrepancies in the categorization across cohorts

In [8]:
nhanes.harmonize_categorization()

## Inspect variables

Inpecting plots before transformations

In [9]:
nhanes.plot_variables(plot_paths)

Generating a 3 page PDF for 30 variables
Generating a 1 page PDF for 4 variables
Generating a 28 page PDF for 326 variables
Generating a 3 page PDF for 30 variables
Generating a 1 page PDF for 4 variables
Generating a 28 page PDF for 326 variables
Generating a 3 page PDF for 30 variables
Generating a 1 page PDF for 4 variables
Generating a 28 page PDF for 326 variables
Generating a 3 page PDF for 30 variables
Generating a 1 page PDF for 4 variables
Generating a 28 page PDF for 326 variables


## Transform continuous variables

- Log 2 transform to all continuous variables: those with negative skews are reflected first, and then reflected back
- Normalize with mean zero and unit variance

In [10]:
neg_skewed = nhanes.transform_continuous_log2()
nhanes.zscore_normalization()

-----Log2 transforming 326 variables-----
-----Log2 transforming 326 variables-----
-----Log2 transforming 326 variables-----
-----Log2 transforming 326 variables-----
-----Scaling 326 variables-----
-----Scaling 326 variables-----
-----Scaling 326 variables-----
-----Scaling 326 variables-----


## Remove outliers

Only for continuous variables

In [11]:
nhanes.remove_continuous_outliers()

Running remove_outliers
--------------------------------------------------------------------------------
Removing outliers from 4,724 observations of 326 continuous variables with values more than 3 standard deviations from the mean
	Removed 0 low and 5 high outliers from LBX187 (outside -3.00 to 3.00)
	Removed 9 low and 0 high outliers from SMD100NI (outside -3.00 to 3.00)
	Removed 0 low and 5 high outliers from URXMHH (outside -3.00 to 3.00)
	Removed 0 low and 9 high outliers from LBX156 (outside -3.00 to 3.00)
	Removed 44 low and 0 high outliers from LBDRUIU (outside -3.00 to 3.00)
	Removed 37 low and 2 high outliers from DR1TVB6 (outside -3.00 to 3.00)
	Removed 18 low and 5 high outliers from DR1TFDFE (outside -3.00 to 3.00)
	Removed 0 low and 61 high outliers from THIAMIN_mg (outside -3.00 to 3.00)
	Removed 0 low and 4 high outliers from URXUPB (outside -3.00 to 3.00)
	Removed 0 low and 13 high outliers from URXUTU (outside -3.00 to 3.00)
	Removed 0 low and 3 high outliers from UR

Make another pass for the categorization and filters to remove variables which output was NA due to outlier removal

In [12]:
nhanes.categorize_variables()
nhanes.make_unknown_continuous()
nhanes.clarite_filters()

-----Categorizing in Discovery females cohort-----
Running categorize
--------------------------------------------------------------------------------
1 of 360 variables (0.28%) are classified as constant (1 unique value).
32 of 360 variables (8.89%) are classified as binary (2 unique values).
13 of 360 variables (3.61%) are classified as categorical (3 to 6 unique values).
298 of 360 variables (82.78%) are classified as continuous (>= 15 unique values).
0 of 360 variables (0.00%) were dropped.
16 of 360 variables (4.44%) were not categorized and need to be set manually.
	16 variables had between 6 and 15 unique values
	0 variables had >= 15 values but couldn't be converted to continuous (numeric) values

-----Removing constant variables-----
We will remove 1 constant variables
-----Categorizing in Discovery males cohort-----
Running categorize
--------------------------------------------------------------------------------
1 of 360 variables (0.28%) are classified as constant (1 uniqu

In [13]:
nhanes.plot_variables(plotpath=plot_paths,
                      suffix='Transformed',
                      only_continuous=True)

Generating a 26 page PDF for 307 variables
Generating a 26 page PDF for 307 variables
Generating a 26 page PDF for 307 variables
Generating a 26 page PDF for 307 variables


## Save cleaned data

In [14]:
nhanes.save_pheno_expo_list(paths[2])

In [15]:
nhanes.print_how_many_types()
nhanes.save_data(paths[2])

There are 8 covariates, 59 phenotypes, and 275 exposures
